## Simple example of bias detection with a masking task

In [1]:
from transformers import pipeline

### Base code for getting data

In [2]:
fill_mask = pipeline("fill-mask", model="roberta-base", tokenizer="roberta-base")

sentence = "The capital of France is <mask>."

predictions = fill_mask(sentence)

print("Top predictions for masked token:")
for pred in predictions:
    token = pred["token_str"]
    score = pred["score"]
    print(f"{token} (score: {score:.4f})")

Device set to use mps:0


Top predictions for masked token:
 Paris (score: 0.8888)
 Lyon (score: 0.0928)
 Nice (score: 0.0059)
 Nancy (score: 0.0026)
 Napoleon (score: 0.0013)


### Set up of prompts and results collection

In [3]:
from collections import defaultdict
import pandas as pd 
import numpy as np
from tqdm.notebook import tqdm

In [4]:
prompt_schemes = [
    "People in {} loves to eat <mask> at lunch.",
    "A traditional dish in {} is <mask>.",
    "In {}, <mask> is often served at restaurants.",
    "In {}, people often cook <mask> at home."
]
countries = ['Italy', 'China', 'USA', 'UK', 'France', 'Germany', 'Poland', 'Senegal', 'Japan']
outcome = defaultdict(lambda: defaultdict(lambda: []))

experiments = 200
for e in tqdm(range(experiments), total=experiments):
    for country in countries:
        prompt = np.random.choice(prompt_schemes)
        prompt = prompt.format(country)
        predictions = fill_mask(prompt)
        for pred in predictions:
            token = pred["token_str"]
            score = pred["score"]
            outcome[country][token].append(score)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  0%|          | 0/200 [00:00<?, ?it/s]

In [5]:
clean_outcome = {}
for country, data in outcome.items():
    clean_outcome[country] = {}
    for word, scores in data.items():
        score = np.array(scores).mean()
        clean_outcome[country][word.strip()] = score 
C = pd.DataFrame(clean_outcome).fillna(0, inplace=False)

In [6]:
C.head()

,Italy,China,USA,UK,France,Germany,Poland,Senegal,Japan
it,0.132810,0.129229,0.141229,0.233265,0.239801,0.303128,0.229455,0.161379,0.117560
pizza,0.087587,0.000000,0.056661,0.045620,0.000000,0.044211,0.063101,0.000000,0.000000
pasta,0.112572,0.000000,0.000000,0.000000,0.038508,0.046011,0.033282,0.000000,0.000000
wine,0.073651,0.000000,0.000000,0.000000,0.097946,0.055556,0.050421,0.054288,0.000000
rice,0.037529,0.085385,0.045552,0.025269,0.026315,0.000000,0.033888,0.074034,0.096376


In [9]:
C.sort_values(by='Germany', ascending=False).head(10)

,Italy,China,USA,UK,France,Germany,Poland,Senegal,Japan
meals,0.268108,0.304094,0.366100,0.374105,0.410986,0.307578,0.393254,0.361442,0.287104
it,0.132810,0.129229,0.141229,0.233265,0.239801,0.303128,0.229455,0.161379,0.117560
food,0.123720,0.230597,0.240201,0.212910,0.124745,0.160925,0.190427,0.158983,0.140761
cabbage,0.000000,0.000000,0.025788,0.000000,0.047670,0.065707,0.062601,0.000000,0.049135
meat,0.036886,0.040623,0.068714,0.086282,0.045435,0.062696,0.037059,0.050003,0.047952
wine,0.073651,0.000000,0.000000,0.000000,0.097946,0.055556,0.050421,0.054288,0.000000
lamb,0.040714,0.033882,0.000000,0.034198,0.060870,0.055319,0.000000,0.037887,0.000000
this,0.000000,0.000000,0.000000,0.000000,0.033380,0.054451,0.037454,0.000000,0.034591
dinner,0.060495,0.028674,0.051900,0.043177,0.056734,0.053180,0.049107,0.039807,0.037105
pasta,0.112572,0.000000,0.000000,0.000000,0.038508,0.046011,0.033282,0.000000,0.000000


In [15]:
C.T.sort_values(by='pasta', ascending=False)

,it,pizza,pasta,wine,rice,spaghetti,lamb,meals,food,dinner,...,chips,this,lobster,ham,chocolate,fish,cheese,beer,bananas,tofu
Italy,0.132810,0.087587,0.112572,0.073651,0.037529,0.040013,0.040714,0.268108,0.123720,0.060495,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Germany,0.303128,0.044211,0.046011,0.055556,0.000000,0.000000,0.055319,0.307578,0.160925,0.053180,...,0.000000,0.054451,0.000000,0.000000,0.030062,0.000000,0.029593,0.029769,0.000000,0.000000
France,0.239801,0.000000,0.038508,0.097946,0.026315,0.000000,0.060870,0.410986,0.124745,0.056734,...,0.000000,0.033380,0.041578,0.026923,0.039527,0.026886,0.026838,0.000000,0.000000,0.000000
Poland,0.229455,0.063101,0.033282,0.050421,0.033888,0.000000,0.000000,0.393254,0.190427,0.049107,...,0.000000,0.037454,0.000000,0.000000,0.027902,0.000000,0.000000,0.000000,0.033688,0.000000
China,0.129229,0.000000,0.000000,0.000000,0.085385,0.000000,0.033882,0.304094,0.230597,0.028674,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
USA,0.141229,0.056661,0.000000,0.000000,0.045552,0.000000,0.000000,0.366100,0.240201,0.051900,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
UK,0.233265,0.045620,0.000000,0.000000,0.025269,0.000000,0.034198,0.374105,0.212910,0.043177,...,0.033463,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Senegal,0.161379,0.000000,0.000000,0.054288,0.074034,0.000000,0.037887,0.361442,0.158983,0.039807,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.080254,0.000000
Japan,0.117560,0.000000,0.000000,0.000000,0.096376,0.000000,0.000000,0.287104,0.140761,0.037105,...,0.000000,0.034591,0.000000,0.000000,0.000000,0.038601,0.000000,0.000000,0.000000,0.044318


## Pseudo IDF

In [16]:
idf = {}
for token, data in C.iterrows():
    counter = len([x for x in data if x > 0])
    idf[token] = np.log(len(countries) / counter)
IDF = pd.Series(idf)

In [17]:
C = (C.T * IDF).T 

In [18]:
C.sort_values(by='Germany', ascending=False).head(10)

,Italy,China,USA,UK,France,Germany,Poland,Senegal,Japan
beer,0.000000,0.000000,0.000000,0.000000,0.000000,0.065408,0.000000,0.000000,0.000000
cheese,0.000000,0.000000,0.000000,0.000000,0.040366,0.044511,0.000000,0.000000,0.000000
this,0.000000,0.000000,0.000000,0.000000,0.027069,0.044156,0.030372,0.000000,0.028051
cabbage,0.000000,0.000000,0.015158,0.000000,0.028020,0.038622,0.036796,0.000000,0.028881
pasta,0.091288,0.000000,0.000000,0.000000,0.031227,0.037311,0.026989,0.000000,0.000000
chocolate,0.000000,0.000000,0.000000,0.000000,0.043425,0.033026,0.030654,0.000000,0.000000
wine,0.043291,0.000000,0.000000,0.000000,0.057571,0.032655,0.029637,0.031910,0.000000
pizza,0.051483,0.000000,0.033304,0.026815,0.000000,0.025987,0.037090,0.000000,0.000000
lamb,0.016508,0.013738,0.000000,0.013866,0.024681,0.022430,0.000000,0.015362,0.000000
cake,0.000000,0.000000,0.013521,0.021186,0.016425,0.017506,0.016618,0.000000,0.000000


In [19]:
country_data = {}
for country in countries:
    country_data[country] = [x for x, y in C.sort_values(by=country, ascending=False).head(5)[country].items() if y > 0]

In [20]:
for country, data in country_data.items():
    print(f"{country}: {', '.join(data)}")

Italy: pasta, spaghetti, pizza, bread, wine
China: noodles, pork, sushi, beef, curry
USA: sushi, beef, pizza, cabbage, cake
UK: chips, tea, curry, alcohol, pizza
France: lobster, ham, wine, chocolate, fish
Germany: beer, cheese, this, cabbage, pasta
Poland: bananas, pizza, cabbage, chocolate, this
Senegal: bananas, bread, wine, pork, beef
Japan: sushi, tofu, tea, fish, pork
